# Tutorial - Three-Phase Induction Motor (TPIM)
======================

This is a basic tutorial on how to use the `MotorElement` class to simulate a **Three-Phase Induction Motor (TPIM)** in ROSS. Before starting this tutorial, be sure you're already familiar with the ROSS library.

`MotorElement` models the electromechanical behavior of a TPIM using a synchronous reference frame for rotor flux orientation. The stator/rotor flux linkages and the shaft speed are integrated in time with a 4th-order Runge-Kutta method, from which the electric torque, shaft speed, stator currents and stator voltages are obtained.

The motor can be driven by three different electrical sources:
- An ideal three-phase AC source (`SourceAC`), which can optionally include voltage harmonics and voltage unbalance, used through `.run_with_AC_source()`;
- A variable frequency drive (`InverterVF`) using Space Vector PWM (SVPWM) modulation with scalar V/f speed control, used through `.run_with_inverter_vf()`;
- A variable frequency drive (`InverterFOC`) using SVPWM modulation with indirect Field-Oriented Control (iFOC), a closed-loop speed/current control scheme, used through `.run_with_inverter_foc()`.

It is important to note that `MotorElement` is **not** a structural rotor element: it does not contribute to the rotor's global mass, stiffness, damping, or gyroscopic matrices assembled by `Rotor`. The node `n` only identifies where the motor torque is applied on the shaft (torsional DOF). To couple the motor with a rotor model, use `.run_with_AC_source()` / `.run_with_inverter_vf()` / `.run_with_inverter_foc()` to obtain `electric_torque`, then apply it as the excitation in `.run_time_response()`.

**This tutorial is split into three notebooks**, to keep each one at a manageable size:
- **tutorial_part_6.ipynb** (this notebook): Section 1 (Quick Start with `motor_example`) and Section 2 (Custom Instantiation with Voltage Harmonics and Unbalance), both driven by an ideal `SourceAC`;
- **tutorial_part_7.ipynb**: Section 3, the `InverterVF` (V/f) variable frequency drive;
- **tutorial_part_8.ipynb**: Section 4, the `InverterFOC` (indirect Field-Oriented Control) drive, plus a comparison between V/f and FOC speed control.

Each notebook is self-contained and can be run independently.

# Section 1: Quick Start with `motor_example`

## 1.1 MotorElement Class

ROSS provides the helper function `motor_example()`, which returns an instance of `MotorElement` pre-configured for a **1.5 hp / 127 V / 60 Hz / 4-pole** motor with a nominal speed of **1710 RPM**. This is a convenient starting point to explore the class API.

In [ ]:
import ross as rs
import numpy as np

from ross import motor_example
from ross.units import Q_

# Make sure the default renderer is set to 'notebook' for inline plots in Jupyter
import plotly.io as pio
import plotly.graph_objects as go

pio.renderers.default = "notebook"

In [ ]:
motor = motor_example()
motor

## 1.2 Running with an Ideal AC Source

The `.run_with_AC_source()` method simulates the motor connected to an ideal three-phase voltage source (`SourceAC`). The main arguments are:

- `t`: array of time points at which the solution is stored [s];
- `load_torque_entrance_time`: time at which the load torque is applied to the shaft [s];
- `load_torque_ratio`: fraction of the nominal torque (`Tnom`) applied at the entrance time (1.0 = 100% of `Tnom`).

It returns a `MotorResponseResults` object containing the time histories of `electric_torque`, `load_torque`, `speed`, `currents` and `voltages`.

In [ ]:
dt = 1e-3
tf = 3.0
t = np.arange(0, tf + dt, dt)

results = motor.run_with_AC_source(
    t,
    load_torque_entrance_time=1.5,
    load_torque_ratio=1.0,
)

## 1.3 Visualizing Results

The `MotorResponseResults` object provides several plotting methods, all returning Plotly figures.

### Electromagnetic Torque

In [ ]:
results.plot_torque().show()

### Rotor Speed

In [ ]:
results.plot_speed().show()

### Stator Phase Currents

The `reference_frame` argument selects between the natural three-phase frame (`a-b-c`), the Clarke transform (`alpha-beta`), and the Park transform (`d-q`).

In [ ]:
results.plot_phase_currents(reference_frame="a-b-c").show()

### Stator Phase Voltages

In [ ]:
results.plot_phase_voltages().show()

### Stator Line Voltages

The line voltages (AB, BC, CA) are also available through `.plot_line_voltages()`.

In [ ]:
results.plot_line_voltages().show()

# Section 2: Custom Instantiation with Voltage Harmonics and Unbalance

## 2.1 Instantiating the Motor

`MotorElement` can also be instantiated by directly informing the nominal and circuit parameters of the machine. In this section, the motor corresponds to a **1.5 hp** TPIM, fed at **127 V / 60 Hz**, with **4 poles** and a nominal speed of **1710 RPM**.

| Parameter | Value | Description |
|---|---|---|
| `power_nom` | `Q_(1.5, "hp")` | Nominal power |
| `voltage_nom` | 127 V | Nominal voltage |
| `speed_nom` | `Q_(1710, "RPM")` | Nominal machine speed |
| `frequency_nom` | `Q_(60, "Hz")` | Nominal frequency |
| `n_poles` | 4 | Number of poles |
| `stator_resistance` / `rotor_resistance` | 2.5 / 1.8 Ohm | Stator / rotor resistance |
| `stator_reactance` / `rotor_reactance` | 1.3 / 1.3 Ohm | Stator / rotor self-reactance |
| `mutual_reactance` | 43.08 Ohm | Mutual reactance |
| `Ip_motor` | 0.0372 kg.m² | Polar moment of inertia related to motor axis |
| `voltage_net` / `frequency_net` | 127 V / `Q_(60, "Hz")` | Power supply voltage and frequency |

In [ ]:
motor2 = rs.MotorElement(
    n=0,
    tag="TPIM_1p5hp",
    power_nom=Q_(1.5, "hp"),
    voltage_nom=127,
    speed_nom=Q_(1710, "RPM"),
    frequency_nom=Q_(60.0, "Hz"),
    n_poles=4,
    stator_resistance=2.5,
    rotor_resistance=1.8,
    stator_reactance=1.3,
    rotor_reactance=1.3,
    mutual_reactance=43.08,
    Ip_motor=0.0372,
    viscosity_coeff=0.0,
    Ip_load=0.0,
    voltage_net=127,
    frequency_net=Q_(60.0, "Hz"),
)
motor2

## 2.2 Voltage Harmonics and Unbalance

Voltage harmonics and voltage unbalance are configured by passing the `harmonics` and `unbalances` dictionaries to `.run_with_AC_source()`.

`harmonics` accepts:
- `enable`: enables the harmonic content of the source;
- `orders`: list with the harmonic orders (e.g. 5th and 7th);
- `amplitudes`: list with the harmonic amplitudes, as a percentage of `voltage_net`.

`unbalances` accepts:
- `enable`: enables the voltage unbalance;
- `voltage_percent`: per-phase voltage magnitude deviation [%], in the order [phase A, phase B, phase C];
- `angle_deviation`: per-phase voltage angle deviation, in the order [phase A, phase B, phase C].

In this example, the 5th harmonic (300 Hz) is set with 3% amplitude and the 7th harmonic (420 Hz) with 2% amplitude. The voltage unbalance applied is:

| Phase | Voltage deviation | Angle deviation |
|---|---|---|
| A | -1% | +1° |
| B | +2% | 0° |
| C | +3% | -2° |

In [ ]:
dt = 1e-3
tf = 3.0
t2 = np.arange(0, tf + dt, dt)

results2 = motor2.run_with_AC_source(
    t2,
    load_torque_entrance_time=1.5,
    load_torque_ratio=1.0,
    harmonics={
        "enable": True,
        "orders": [5, 7],
        "amplitudes": [3.0, 2.0],
    },
    unbalances={
        "enable": True,
        "voltage_percent": [-1.0, 2.0, 3.0],
        "angle_deviation": Q_([1.0, 0.0, -2.0], "deg"),
    },
)

## 2.3 Time-Domain Results

### Electromagnetic Torque

In [ ]:
results2.plot_torque().show()

### Rotor Speed

In [ ]:
results2.plot_speed().show()

### Stator Phase Currents

In [ ]:
results2.plot_phase_currents(reference_frame="a-b-c").show()

### Stator Phase Voltages

In [ ]:
results2.plot_phase_voltages().show()

## 2.4 Frequency-Domain Results (FFT)

Every plotting method accepts `domain="frequency"`, which displays the windowed FFT of the corresponding signal instead of its time history. This makes it possible to identify the injected voltage harmonics in the currents and voltages. Since `SourceAC` is an ideal supply (no IGBT switching involved), no `frequency_range` is applied here and the full computed spectrum is shown.

### FFT of the Electromagnetic Torque

In [ ]:
results2.plot_torque(domain="frequency").show()

### FFT of the Stator Phase Currents

The 5th and 7th harmonic components should be visible in the spectrum, as a consequence of the harmonics enabled in the source.

In [ ]:
results2.plot_phase_currents(domain="frequency").show()

### FFT of the Stator Phase Voltages

In [ ]:
results2.plot_phase_voltages(domain="frequency").show()

---
Continue to **tutorial_part_7.ipynb** for the Variable Frequency Drive (`InverterVF`, V/f control) example.